Decoder Only Plus Ollama And Encoder Vizualization

In [2]:
import pandas as pd
import numpy as np
import re
import ollama
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from umap import UMAP

EMBEDDING_MODEL = 'BAAI/bge-m3'
TRANSLATOR_MODEL = "qwen2.5:7b"
EVALUATOR_MODEL = "qwen2.5:7b" 

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s\-\'àâçéèêëîïôûùüÿñæœ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def generate_translation(text, model):
    prompt = f"Translate the following French sentence into English. Only output the translation, nothing else.\n\nFrench: {text}\nEnglish:"
    try:
        response = ollama.generate(model=model, prompt=prompt)
        return response['response'].strip()
    except Exception as e:
        return ""

def evaluate_translation_category(source, reference, candidate, idiom, model):
    """Juge la qualité de la traduction selon la taxonomie IDIOMEVAL"""
    prompt = f"""
    ## Task
    Analyze the French idiom "{idiom}", the source sentence, the Reference translation, and the Candidate translation.
    Classify the Candidate translation into ONE category.
    
    ## Categories
    - Good Translation: Correct figurative meaning.
    - Literal Translation: Word-for-word, ignores figurative meaning.
    - Mistranslation: Wrong meaning.
    - Partial Translation: Misses part of the meaning.
    - Unnatural: Correct meaning but awkward phrasing.
    - Hallucination: Completely unrelated content.

    ## Input
    * Source: "{source}"
    * Reference: "{reference}"
    * Candidate: "{candidate}"

    Respond ONLY with a JSON object: {{"category": "Category Name"}}
    """
    try:
        response = ollama.generate(model=model, prompt=prompt, format='json')
        return json.loads(response['response']).get('category', 'Unknown')
    except:
        return "Evaluation Error"

print("Chargement des données...")
df = pd.read_csv('../large_idiom_set_fr_eng.csv')

df['clean_french'] = df['original_text'].apply(clean_text)
df['clean_english'] = df['text'].apply(clean_text)



Chargement des données...


In [7]:
import os
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
tqdm.pandas()  
model = SentenceTransformer(EMBEDDING_MODEL)
# Get all unique idioms from the dataset
unique_idioms = df['contains_idioms'].unique()
print(f"Found {len(unique_idioms)} unique idioms")

# Store all results
all_results_combined = []
all_similarity_results = []

# Loop through each idiom
for idiom in unique_idioms:
    if pd.isna(idiom):
        continue
    
    print(f"\n{'='*60}")
    print(f"Processing idiom: {idiom}")
    print(f"{'='*60}")
    
    # Filter for this idiom
    df_target = df[df['contains_idioms'].astype(str).str.contains(idiom, na=False)].copy()
    print(f"Found {len(df_target)} sentences with '{idiom}'")
    
    if len(df_target) == 0:
        continue
    
    df_target = df_target.reset_index(drop=True)
    
    # --- GENERATION & EVALUATION (same as before) ---
    df_target['generated_translation'] = df_target['clean_french'].progress_apply(
        lambda x: generate_translation(x, TRANSLATOR_MODEL)
    )
    
    df_target['eval_category'] = df_target.progress_apply(
        lambda row: evaluate_translation_category(
            row['clean_french'], 
            row['clean_english'], 
            row['generated_translation'], 
            idiom, 
            EVALUATOR_MODEL
        ), axis=1
    )
    
    # --- EMBEDDINGS & SIMILARITY ---
    eng_embeddings = model.encode(df_target['clean_english'].tolist(), show_progress_bar=False)
    gen_embeddings = model.encode(df_target['generated_translation'].tolist(), show_progress_bar=False)
    fr_embeddings = model.encode(df_target['clean_french'].tolist(), show_progress_bar=False)
    
    # Compute similarities for this idiom
    similarity_results = []
    data_records = []
    
    for idx in range(len(df_target)):
        french_emb = fr_embeddings[idx].reshape(1, -1)
        eng_emb = eng_embeddings[idx].reshape(1, -1)
        gen_emb = gen_embeddings[idx].reshape(1, -1)
        
        sim_fr_eng = cosine_similarity(french_emb, eng_emb)[0][0]
        sim_fr_gen = cosine_similarity(french_emb, gen_emb)[0][0]
        
        # Similarity record
        similarity_results.append({
            'idiom': idiom,
            'idiom_id': idx,
            'french_text': df_target.loc[idx, 'clean_french'],
            'english_reference': df_target.loc[idx, 'clean_english'],
            'generated_translation': df_target.loc[idx, 'generated_translation'],
            'eval_category': df_target.loc[idx, 'eval_category'],
            'similarity_french_to_english': sim_fr_eng,
            'similarity_french_to_generated': sim_fr_gen,
            'similarity_gap': sim_fr_eng - sim_fr_gen
        })
        
        # Embedding record (for Atlas visualization)
        data_records.append({
            'idiom': idiom,
            'idiom_id': idx,
            'text': df_target.loc[idx, 'clean_english'],
            'point_type': 'English Reference (Gold)',
            'category': 'Reference',
            'embedding': eng_embeddings[idx]
        })
        data_records.append({
            'idiom': idiom,
            'idiom_id': idx,
            'text': df_target.loc[idx, 'generated_translation'],
            'point_type': 'Generated Translation',
            'category': df_target.loc[idx, 'eval_category'],
            'embedding': gen_embeddings[idx]
        })
        data_records.append({
            'idiom': idiom,
            'idiom_id': idx,
            'text': df_target.loc[idx, 'clean_french'],
            'point_type': 'French Source',
            'category': 'Source',
            'embedding': fr_embeddings[idx]
        })
    
    # Save per-idiom CSV
    df_sim = pd.DataFrame(similarity_results)
    csv_filename = f"idiom_similarity_{idiom.replace(' ', '_')}.csv"
    df_sim.to_csv(f"Scores/{csv_filename}", index=False)
    print(f"✓ Saved: {csv_filename}")
    
    # Save per-idiom parquet (for Atlas)
    df_records = pd.DataFrame(data_records)
    if len(df_records) > 5:
        all_embeddings = np.stack(df_records['embedding'].values)
        reducer = UMAP(n_components=2, metric='cosine', random_state=42)
        projections = reducer.fit_transform(all_embeddings)
        
        df_records['x'] = projections[:, 0]
        df_records['y'] = projections[:, 1]
        
        parquet_filename = f"idiom_eval_{idiom.replace(' ', '_')}.parquet"
        df_records.to_parquet(f"Vizualization/{parquet_filename}")
        print(f"✓ Saved: {parquet_filename}")
    
    # Append to combined results
    all_results_combined.append(df_records)
    all_similarity_results.append(df_sim)

# Create combined files
print("\n" + "="*60)
print("Creating combined files...")
print("="*60)

df_all_results = pd.concat(all_results_combined, ignore_index=True)
df_all_similarity = pd.concat(all_similarity_results, ignore_index=True)

df_all_results.to_parquet('all_idioms_combined.parquet')
df_all_similarity.to_csv('all_idioms_similarity_combined.csv', index=False)

print("✓ Saved: all_idioms_combined.parquet")
print("✓ Saved: all_idioms_similarity_combined.csv")

Found 181 unique idioms

Processing idiom: ah la vache
Found 10 sentences with 'ah la vache'


100%|██████████| 10/10 [00:01<00:00,  5.37it/s]


✓ Saved: idiom_similarity_ah_la_vache.csv


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✓ Saved: idiom_eval_ah_la_vache.parquet

Processing idiom: aller droit à le but
Found 101 sentences with 'aller droit à le but'


...

✓ Saved: idiom_eval_être_à_le_ouest.parquet

Processing idiom: être à le taquet
Found 22 sentences with 'être à le taquet'


100%|██████████| 22/22 [00:04<00:00,  5.19it/s]


✓ Saved: idiom_similarity_être_à_le_taquet.csv
✓ Saved: idiom_eval_être_à_le_taquet.parquet

Creating combined files...


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✓ Saved: all_idioms_combined.parquet
✓ Saved: all_idioms_similarity_combined.csv


In Terminal : embedding-atlas Code2Qwen/embeddings_full.parquet --x x --y y --text text